In [0]:
from pyspark.sql import functions as F

pipeline_df = spark.sql("""
SELECT
    DATE(snapshot_timestamp) as run_date,
    MAX(bronze_job_count) as bronze_job_count,
    MAX(silver_job_count) as silver_job_count,
    MAX(extracted_skills) as extracted_skills,
    MAX(unique_skills) as unique_skills
FROM skillpulse.monitoring.pipeline_run_history
GROUP BY DATE(snapshot_timestamp)
ORDER BY run_date DESC
""")

In [0]:
spark.sql("""
DELETE FROM skillpulse.monitoring.alert_events
WHERE DATE(alert_time) = CURRENT_DATE()
AND alert_type IN (
    'PIPELINE_MISMATCH',
    'JOB_VOLUME_DROP',
    'SKILL_EXTRACTION_DROP',
    'NEW_SKILL'
)
""")

DataFrame[num_affected_rows: bigint]

In [0]:
def create_alert(
    alert_type,
    severity,
    message
):

    spark.sql(f"""
    INSERT INTO skillpulse.monitoring.alert_events
    VALUES (
        current_timestamp(),
        '{alert_type}',
        '{severity}',
        '{message}'
    )
    """)

In [0]:
latest = pipeline_df.limit(1).collect()[0]

if latest["bronze_job_count"] != latest["silver_job_count"]:

    create_alert(
        "PIPELINE_MISMATCH",
        "CRITICAL",
        f"Bronze count {latest['bronze_job_count']} "
        f"does not match Silver count "
        f"{latest['silver_job_count']}"
    )

In [0]:
latest_two = pipeline_df.limit(2).collect()

if len(latest_two) == 2:

    today_jobs = latest_two[0]["bronze_job_count"]

    yesterday_jobs = latest_two[1]["bronze_job_count"]

    if today_jobs < (yesterday_jobs * 0.8):

        create_alert(
            "JOB_VOLUME_DROP",
            "WARNING",
            f"Job volume dropped from "
            f"{yesterday_jobs} to {today_jobs}"
        )

In [0]:
if len(latest_two) == 2:

    today_skills = latest_two[0]["extracted_skills"]

    yesterday_skills = latest_two[1]["extracted_skills"]

    if today_skills < (yesterday_skills * 0.8):

        create_alert(
            "SKILL_EXTRACTION_DROP",
            "WARNING",
            f"Extracted skills dropped from "
            f"{yesterday_skills} to {today_skills}"
        )

In [0]:
skill_history = spark.table(
    "skillpulse.gold.gold_skill_daily_snapshot"
)

latest_date = (
    skill_history
    .agg(F.max("snapshot_date"))
    .collect()[0][0]
)

previous_date = (
    skill_history
    .filter(
        F.col("snapshot_date") < latest_date
    )
    .agg(F.max("snapshot_date"))
    .collect()[0][0]
)

latest_skills = (
    skill_history
    .filter(
        F.col("snapshot_date") == latest_date
    )
    .select("skill")
)

previous_skills = (
    skill_history
    .filter(
        F.col("snapshot_date") == previous_date
    )
    .select("skill")
)

new_skills = (
    latest_skills
    .subtract(previous_skills)
)

for row in new_skills.collect():

    create_alert(
        "NEW_SKILL",
        "INFO",
        f"New skill detected: {row['skill']}"
    )

In [0]:
spark.sql("""
DELETE FROM skillpulse.monitoring.alert_events
WHERE alert_type = 'PIPELINE_HEALTH'
AND DATE(alert_time) = CURRENT_DATE()
""")

DataFrame[num_affected_rows: bigint]

In [0]:
create_alert(
    "PIPELINE_HEALTH",
    "INFO",
    "Daily monitoring checks completed successfully"
)

In [0]:
%sql
SELECT *
FROM skillpulse.monitoring.alert_events
ORDER BY alert_time DESC;

alert_time,alert_type,severity,message
2026-06-11T10:44:46.468Z,PIPELINE_HEALTH,INFO,Daily monitoring checks completed successfully
